## Cell 1 — Imports

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

## Cell 2 — Set paths

In [4]:
csv_path = Path("../data/raw/ecommerce_data_final.csv")
db_path = Path("../data/ecommerce.db")

print(csv_path)
print(db_path)

..\data\raw\ecommerce_data_final.csv
..\data\ecommerce.db


## Cell 3 — Load and inspect product data

In [5]:
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Shape: (903, 7)

Columns:
['product_link', 'title', 'brand', 'price', 'discount', 'avg_rating', 'total_ratings']


,product_link,title,brand,price,discount,avg_rating,total_ratings
0,https://www.flipkart.com/aadi-mesh-lightweight...,Mesh |Lightweight|Comfort|Summer|Trendy|Walkin...,aadi,539,0.73,4.0,43
1,https://www.flipkart.com/campus-alice-running-...,ALICE Running Shoes For Women,CAMPUS,1063,0.24,4.3,24914
2,https://www.flipkart.com/shozie-stylish-sneake...,Stylish Sneakers Girls Walking Shoes For Women,Shozie,449,0.55,4.0,22190
3,https://www.flipkart.com/sparx-sl-194-running-...,SL 194 Running Shoes For Women,Sparx,824,0.25,4.2,2804
4,https://www.flipkart.com/campus-camp-eloy-walk...,CAMP ELOY Walking Shoes For Women,CAMPUS,976,0.02,4.3,2770


## Once columns are confirmed: SQLite ingestion

In [6]:
with sqlite3.connect(db_path) as conn:
    df.to_sql(
        "product",
        conn,
        if_exists="replace",
        index=False
    )

In [8]:
## Cell 4 — Validate schema and data

In [9]:
expected_columns = [
    "product_link",
    "title",
    "brand",
    "price",
    "discount",
    "avg_rating",
    "total_ratings"
]

print("Schema correct:", df.columns.tolist() == expected_columns)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

Schema correct: True

Data types:
product_link         str
title                str
brand                str
price              int64
discount         float64
avg_rating       float64
total_ratings      int64
dtype: object

Missing values:
product_link     0
title            0
brand            0
price            0
discount         0
avg_rating       0
total_ratings    0
dtype: int64

Duplicate rows: 0


## Cell 5 — Create the SQLite database

In [11]:
with sqlite3.connect(db_path) as conn:
    df.to_sql(
        name="product",
        con=conn,
        if_exists="replace",
        index=False
    )

print("Product data successfully written to SQLite.")

Product data successfully written to SQLite.


## Cell 6 — Verify the database

In [12]:
with sqlite3.connect(db_path) as conn:
    result = pd.read_sql_query(
        "SELECT * FROM product LIMIT 5",
        conn
    )

display(result)

,product_link,title,brand,price,discount,avg_rating,total_ratings
0,https://www.flipkart.com/aadi-mesh-lightweight...,Mesh |Lightweight|Comfort|Summer|Trendy|Walkin...,aadi,539,0.73,4.0,43
1,https://www.flipkart.com/campus-alice-running-...,ALICE Running Shoes For Women,CAMPUS,1063,0.24,4.3,24914
2,https://www.flipkart.com/shozie-stylish-sneake...,Stylish Sneakers Girls Walking Shoes For Women,Shozie,449,0.55,4.0,22190
3,https://www.flipkart.com/sparx-sl-194-running-...,SL 194 Running Shoes For Women,Sparx,824,0.25,4.2,2804
4,https://www.flipkart.com/campus-camp-eloy-walk...,CAMP ELOY Walking Shoes For Women,CAMPUS,976,0.02,4.3,2770


## Cell 7 — Check row count

In [13]:
with sqlite3.connect(db_path) as conn:
    count_df = pd.read_sql_query(
        "SELECT COUNT(*) AS total_products FROM product",
        conn
    )

display(count_df)

,total_products
0,903


## Cell 8 — Find Nike products

In [14]:
query = """
SELECT *
FROM product
WHERE LOWER(brand) LIKE '%nike%'
LIMIT 10;
"""

with sqlite3.connect(db_path) as conn:
    nike_df = pd.read_sql_query(query, conn)

display(nike_df)

,product_link,title,brand,price,discount,avg_rating,total_ratings
0,https://www.flipkart.com/nike-w-revolution-6-n...,W NIKE REVOLUTION 6 NN Running Shoes For Women,NIKE,2469,0.33,3.6,229
1,https://www.flipkart.com/nike-running-shoes-wo...,Running Shoes For Women,NIKE,5349,0.37,4.8,4
2,https://www.flipkart.com/nike-downshifter-12-r...,Downshifter 12 Running Shoes For Women,NIKE,3595,0.10,4.2,1398
3,https://www.flipkart.com/nike-revolution-5-run...,Revolution 5 Running Shoes For Women,NIKE,2786,0.24,4.2,1275
4,https://www.flipkart.com/nike-revolution-6-run...,Revolution 6 Running Shoes For Women,NIKE,2882,0.22,4.3,2035
5,https://www.flipkart.com/nike-escape-run-2-run...,Escape Run 2 Running Shoes For Women,NIKE,5217,0.40,4.7,3
6,https://www.flipkart.com/nike-pegasus-40-runni...,Pegasus 40 Running Shoes For Women,NIKE,7137,0.40,4.8,24
7,https://www.flipkart.com/nike-quest-4-running-...,Quest 4 Running Shoes For Women,NIKE,3597,0.40,4.3,152
8,https://www.flipkart.com/nike-experience-run-1...,Experience Run 11 Training & Gym Shoes For Women,NIKE,4495,0.10,4.5,30
9,https://www.flipkart.com/nike-wmns-tanjun-runn...,WMNS NIKE TANJUN Running Shoes For Women,NIKE,4249,0.14,4.2,605


## Cell 9 — Products under ₹1000 with good ratings

In [23]:
query = """
SELECT *
FROM product
WHERE price < 1000
AND avg_rating >= 4.0
ORDER BY avg_rating DESC, price ASC
LIMIT 10;
"""

with sqlite3.connect(db_path) as conn:
    result = pd.read_sql_query(query, conn)

display(result)

,product_link,title,brand,price,discount,avg_rating,total_ratings
0,https://www.flipkart.com/avant-glam-walking-sh...,Glam Walking Shoes with Cushioned Insoles and ...,AVANT,799,0.27,5.0,5
1,https://www.flipkart.com/abros-london-walking-...,LONDON Walking Shoes For Women,Abros,976,0.06,4.8,9
2,https://www.flipkart.com/sparx-sl-167-outdoors...,SL 167 Outdoors For Women,Sparx,769,0.30,4.7,3
3,https://www.flipkart.com/sparx-sl-258-walking-...,SL 258 Walking Shoes For Women,Sparx,989,0.10,4.7,11
4,https://www.flipkart.com/sparx-sl-258-walking-...,SL 258 Walking Shoes For Women,Sparx,989,0.10,4.7,11
5,https://www.flipkart.com/impakto-ajanta-celest...,Celestial Walking Shoes For Women,Impakto by Ajanta,599,0.57,4.6,5
6,https://www.flipkart.com/impakto-ajanta-celest...,Celestial Walking Shoes For Women,Impakto by Ajanta,599,0.57,4.6,5
7,https://www.flipkart.com/doctor-extra-soft-mem...,"Memory Foam Walking,Gym,Training,Running,Flexi...",DOCTOR EXTRA SOFT,999,0.50,4.6,25
8,https://www.flipkart.com/doctor-extra-soft-mem...,"Memory Foam Walking,Gym,Training,Running,Flexi...",DOCTOR EXTRA SOFT,999,0.50,4.6,25
9,https://www.flipkart.com/yala-sneakers-women/p...,Sneakers For Women,YALA,399,0.42,4.5,6


## Cell 10 — Load Groq client

In [24]:
import os
import re
from dotenv import load_dotenv
from groq import Groq
from pathlib import Path

load_dotenv(Path("..") / ".env")

GROQ_MODEL = os.getenv("GROQ_MODEL")
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print("Model:", GROQ_MODEL)

Model: openai/gpt-oss-20b


## Cell 11 — Define the database schema prompt

In [26]:
sql_prompt = """
You are an expert at converting natural language questions into SQLite queries.

Database schema:

Table: product

Columns:
- product_link: TEXT
- title: TEXT
- brand: TEXT
- price: INTEGER
- discount: REAL
- avg_rating: REAL
- total_ratings: INTEGER

Rules:
1. Generate only SELECT queries.
2. Always query from the product table.
3. Use SELECT * unless aggregation is required.
4. For brand searches, use case-insensitive matching:
   LOWER(brand) LIKE LOWER('%brand%')
5. Use ORDER BY when the user asks for top, highest, lowest, cheapest, best rated, etc.
6. Limit normal product-list queries to 10 rows unless the user requests another number.
7. Never use INSERT, UPDATE, DELETE, DROP, ALTER, or CREATE.
8. Return only the SQL query inside <SQL></SQL> tags.
"""

## Cell 12 — Generate SQL

In [27]:
def generate_sql_query(question):
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": sql_prompt
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [28]:
question = "Show me top 5 Campus shoes under 1500 rupees"

raw_response = generate_sql_query(question)

print(raw_response)

<SQL>
SELECT *
FROM product
WHERE LOWER(brand) LIKE LOWER('%Campus%')
  AND price < 1500
ORDER BY avg_rating DESC
LIMIT 5;
</SQL>


## Cell 13 — Extract SQL from the response

In [29]:
def extract_sql(response):
    match = re.search(
        r"<SQL>(.*?)</SQL>",
        response,
        re.DOTALL
    )

    if not match:
        return None

    return match.group(1).strip()

In [30]:
sql_query = extract_sql(raw_response)

print(sql_query)

SELECT *
FROM product
WHERE LOWER(brand) LIKE LOWER('%Campus%')
  AND price < 1500
ORDER BY avg_rating DESC
LIMIT 5;


## Cell 14 — Basic SQL safety validation

In [31]:
def is_safe_sql(query):
    if not query:
        return False

    cleaned = query.strip().upper()

    forbidden = [
        "INSERT",
        "UPDATE",
        "DELETE",
        "DROP",
        "ALTER",
        "CREATE",
        "TRUNCATE",
        "REPLACE"
    ]

    if not cleaned.startswith("SELECT"):
        return False

    if any(word in cleaned for word in forbidden):
        return False

    return True

In [32]:
print(is_safe_sql(sql_query))

True


## Cell 15 — Execute the generated SQL

In [33]:
def run_query(query):
    if not is_safe_sql(query):
        raise ValueError("Unsafe SQL query blocked.")

    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(query, conn)

In [34]:
result_df = run_query(sql_query)

display(result_df)

,product_link,title,brand,price,discount,avg_rating,total_ratings
0,https://www.flipkart.com/campus-camp-peanut-ru...,CAMP PEANUT Running Shoes For Women,CAMPUS,862,0.13,4.5,44
1,https://www.flipkart.com/campus-camp-callie-wa...,CAMP CALLIE Walking Shoes For Women,CAMPUS,728,0.27,4.5,24
2,https://www.flipkart.com/campus-boond-running-...,BOOND Running Shoes For Women,CAMPUS,1447,0.19,4.5,228
3,https://www.flipkart.com/campus-caly-casuals-w...,CALY Casuals For Women,CAMPUS,1447,0.03,4.4,30
4,https://www.flipkart.com/campus-camp-shimmer-r...,CAMP SHIMMER Running Shoes For Women,CAMPUS,1027,0.57,4.4,1332


In [35]:
## Cell 16 — Test another query

In [36]:
question = "Show me the 3 cheapest shoes with rating above 4.2"

raw_response = generate_sql_query(question)
sql_query = extract_sql(raw_response)

print("Generated SQL:\n")
print(sql_query)

result_df = run_query(sql_query)

display(result_df)

Generated SQL:

SELECT *
FROM product
WHERE title LIKE '%shoe%'
  AND avg_rating > 4.2
ORDER BY price ASC
LIMIT 3;


,product_link,title,brand,price,discount,avg_rating,total_ratings
0,https://www.flipkart.com/feel-time-mesh-lightw...,Mesh |Lightweight|Comfort|Trendy|Walking|Outdo...,FEEL TIME,249,0.75,4.4,11
1,https://www.flipkart.com/jqr-moon-01-sports-sh...,"MOON-01 Sports shoes, Walking, Trendy, Lightwe...",JQR,481,0.46,4.4,149
2,https://www.flipkart.com/jqr-moon-01-sports-sh...,"MOON-01 Sports shoes, Walking, Trendy, Lightwe...",JQR,503,0.44,4.4,149


## Cell 17 — Convert DataFrame into LLM-friendly data

In [37]:
products = result_df.to_dict(orient="records")

products[:2]

[{'product_link': 'https://www.flipkart.com/feel-time-mesh-lightweight-comfort-trendy-walking-outdoor-daily-use-running-shoes-women/p/itm73b17441f83b3?pid=SHOGWESSPD279DFQ&lid=LSTSHOGWESSPD279DFQZOLMAY&marketplace=FLIPKART&q=sports+shoes+for+women&store=osp%2Fiko%2Fd20&srno=s_10_391&otracker=search&otracker1=search&fm=organic&iid=ef38b3f4-a309-4c3b-96b5-4405db1be86e.SHOGWESSPD279DFQ.SEARCH&ppt=sp&ppn=sp&ssid=n0svq94mcg0000001716205005811&qH=32e27b1148959538',
  'title': 'Mesh |Lightweight|Comfort|Trendy|Walking|Outdoor|Daily Use Running Shoes For Women',
  'brand': 'FEEL TIME ',
  'price': 249,
  'discount': 0.75,
  'avg_rating': 4.4,
  'total_ratings': 11},
 {'product_link': 'https://www.flipkart.com/jqr-moon-01-sports-shoes-walking-trendy-lightweight-trekking-stylish-running-shoes-women/p/itm83a88101b47a1?pid=SHOGSX45KXGN6W44&lid=LSTSHOGSX45KXGN6W44XC1D0H&marketplace=FLIPKART&q=sports+shoes+for+women&store=osp%2Fiko%2Fd20&srno=s_11_417&otracker=search&otracker1=search&fm=organic&iid=

## Cell 18 — Create the comprehension prompt

In [38]:
comprehension_prompt = """
You are an e-commerce assistant.

Answer the user's question using ONLY the product data provided.

Rules:
1. Do not invent products or product links.
2. Use only the rows provided in DATA.
3. For product recommendations, show:
   - product title
   - brand
   - price in INR
   - discount percentage
   - rating
   - product link
4. Keep the answer concise.
5. If DATA is empty, say no matching products were found.
"""

## Cell 19 — Generate the final answer

In [39]:
def generate_product_answer(question, products):
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {
                "role": "system",
                "content": comprehension_prompt
            },
            {
                "role": "user",
                "content": f"""
QUESTION:
{question}

DATA:
{products}
"""
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [40]:
final_answer = generate_product_answer(
    question,
    products
)

print(final_answer)

**Cheapest shoes (rating > 4.2)**  

| Title | Brand | Price (INR) | Discount | Rating | Link |
|-------|-------|-------------|----------|--------|------|
| Mesh | Lightweight | Comfort | Trendy | Walking | Outdoor | Daily Use Running Shoes For Women | FEEL TIME | ₹249 | 75 % | 4.4 | https://www.flipkart.com/feel-time-mesh-lightweight-comfort-trendy-walking-outdoor-daily-use-running-shoes-women/p/itm73b17441f83b3?pid=SHOGWESSPD279DFQ&lid=LSTSHOGWESSPD279DFQZOLMAY&marketplace=FLIPKART&q=sports+shoes+for+women&store=osp%2Fiko%2Fd20&srno=s_10_391&otracker=search&otracker1=search&fm=organic&iid=ef38b3f4-a309-4c3b-96b5-4405db1be86e.SHOGWESSPD279DFQ.SEARCH&ppt=sp&ppn=sp&ssid=n0svq94mcg0000001716205005811&qH=32e27b1148959538 |
| MOON‑01 Sports shoes, Walking, Trendy, Lightweight, Trekking, Stylish Running Shoes For Women | JQR | ₹481 | 46 % | 4.4 | https://www.flipkart.com/jqr-moon-01-sports-shoes-walking-trendy-lightweight-trekking-stylish-running-shoes-women/p/itm83a88101b47a1?pid=SHOGSX45K

## Cell 20 — Build the complete sql_chain

In [41]:
def sql_chain(question):
    # 1. Natural language → SQL
    raw_response = generate_sql_query(question)

    # 2. Extract SQL
    sql_query = extract_sql(raw_response)

    if not sql_query:
        return "Sorry, I couldn't generate a valid SQL query."

    print("Generated SQL:")
    print(sql_query)

    # 3. Validate
    if not is_safe_sql(sql_query):
        return "The generated SQL query was blocked for safety."

    # 4. Execute
    result_df = run_query(sql_query)

    if result_df.empty:
        return "No matching products were found."

    # 5. Convert database rows
    products = result_df.to_dict(orient="records")

    # 6. Rows → final natural-language response
    answer = generate_product_answer(
        question,
        products
    )

    return answer

## Cell 21 — Test the full chain

In [43]:
question = "Suggest me 5 Campus shoes under 15000 rupees with the best ratings"

answer = sql_chain(question)

print("\nFINAL ANSWER:\n")
print(answer)

Generated SQL:
SELECT *
FROM product
WHERE LOWER(brand) LIKE LOWER('%Campus%')
  AND price <= 15000
ORDER BY avg_rating DESC
LIMIT 5;

FINAL ANSWER:

**Campus Shoes (≤ ₹15,000) – Best Rated**

| Product Title | Brand | Price (₹) | Discount | Rating | Link |
|---------------|-------|-----------|----------|--------|------|
| CAMP PEANUT Running Shoes For Women | CAMPUS | 862 | 13 % | 4.5 | https://www.flipkart.com/campus-camp-peanut-running-shoes-women/p/itm8b4c45c149168 |
| CAMP CALLIE Walking Shoes For Women | CAMPUS | 728 | 27 % | 4.5 | https://www.flipkart.com/campus-camp-callie-walking-shoes-women/p/itm4c372eb04ce83 |
| BOOND Running Shoes For Women | CAMPUS | 1,447 | 19 % | 4.5 | https://www.flipkart.com/campus-boond-running-shoes-women/p/itmdfd6f98c927c0 |
| CALY Casuals For Women | CAMPUS | 1,447 | 3 % | 4.4 | https://www.flipkart.com/campus-caly-casuals-women/p/itm4ce556eaa7106 |
| CAMP SHIMMER Running Shoes For Women | CAMPUS | 1,027 | 57 % | 4.4 | https://www.flipkart.com/camp